# SETUP

In [2]:
import sys
sys.path.append("/kaggle/input/datasets/moanlobago/3mry5-spotthemaskdata/3mry5_SpotTheMask/3mry5_SpotTheMask/src")

import torch
from data_loader import MaskDataLoader
from model import MaskClassifier
from trainer import Trainer
from predictor import Predictor
from submitter import Submitter

IMAGES_DIR      = "/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/images/images"
TRAIN_CSV       = "/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/train_labels.csv"
SAMPLE_SUB_CSV  = "/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/SampleSubmission.csv"
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


# Load Data

In [3]:
loader = MaskDataLoader(
    images_dir=IMAGES_DIR,
    train_csv=TRAIN_CSV,
    sample_submission_csv=SAMPLE_SUB_CSV,
    val_split=0.15,
    batch_size=32
)
train_loader = loader.get_train_loader()
val_loader   = loader.get_val_loader()
test_loader  = loader.get_test_loader()
print(f"Train: {len(loader.train_df)} | Val: {len(loader.val_df)} | Test: {len(loader.test_df)}")

Train: 1111 | Val: 197 | Test: 509


# Primal Training EfficientNet
## Train

In [4]:
model   = MaskClassifier(dropout=0.4)
trainer = Trainer(model, DEVICE, lr=1e-4, patience=5)
trainer.fit(train_loader, val_loader, epochs=30)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 135MB/s] 


Epoch 001 | Train Loss: 0.6266 | Val Loss: 0.4679
 Saved best model (val_loss=0.4679)


Epoch 002 | Train Loss: 0.3463 | Val Loss: 0.1862
 Saved best model (val_loss=0.1862)


Epoch 003 | Train Loss: 0.1678 | Val Loss: 0.1195
 Saved best model (val_loss=0.1195)


...

Epoch 011 | Train Loss: 0.0458 | Val Loss: 0.0952


Epoch 012 | Train Loss: 0.0315 | Val Loss: 0.0944


Epoch 013 | Train Loss: 0.0363 | Val Loss: 0.0952


Epoch 014 | Train Loss: 0.0314 | Val Loss: 0.0929


Epoch 015 | Train Loss: 0.0373 | Val Loss: 0.0934
Early stopping at epoch 15.


## Predict & Submit

In [5]:
predictor  = Predictor(model, DEVICE, model_path="models/best_model.pth")
fnames, probs = predictor.predict(test_loader)

submitter = Submitter(output_dir="../submissions")
submitter.save(fnames, probs, filename="submission_v1.csv")


## COMMIT

# git init
# git add .
# git commit -m "feat: baseline EfficientNet-B0 mask classifier with OOP pipeline [v0.1.0]"

Submission saved → ../submissions/submission_v1.csv


'../submissions/submission_v1.csv'

# Finetuning EfficientNet
## Finetune with unfrozen layers

In [7]:
# Lower LR for fine-tuning — critical to avoid destroying pretrained weights
model_v2 = MaskClassifier(dropout=0.4)
model_v2.load_state_dict(torch.load("/kaggle/working/models/best_model.pth"))  # start from best
model_v2.unfreeze_backbone(layers_from_end=3)

trainer_v2 = Trainer(model_v2, DEVICE, lr=3e-5, patience=5)  # 3x lower LR
trainer_v2.best_model_path = "/kaggle/working/models/best_model_v2.pth"
trainer_v2.fit(train_loader, val_loader, epochs=20)

Unfroze last 3 backbone blocks.


Epoch 001 | Train Loss: 0.0332 | Val Loss: 0.0933
 Saved best model (val_loss=0.0933)


Epoch 002 | Train Loss: 0.0409 | Val Loss: 0.0969


Epoch 003 | Train Loss: 0.0249 | Val Loss: 0.0939


Epoch 004 | Train Loss: 0.0191 | Val Loss: 0.0917
 Saved best model (val_loss=0.0917)


...

Epoch 009 | Train Loss: 0.0213 | Val Loss: 0.0929


Epoch 010 | Train Loss: 0.0125 | Val Loss: 0.0936


Epoch 011 | Train Loss: 0.0144 | Val Loss: 0.0907


Epoch 012 | Train Loss: 0.0146 | Val Loss: 0.0902


Epoch 013 | Train Loss: 0.0186 | Val Loss: 0.0934
Early stopping at epoch 13.


## Predict with TTA & Submit

In [8]:
predictor_v2 = Predictor(model_v2, DEVICE, model_path="/kaggle/working/models/best_model_v2.pth")
fnames, probs = predictor_v2.predict(test_loader, tta=True)
submitter = Submitter(output_dir="/kaggle/working/submissions")
submitter.save(fnames, probs, filename="submission_v2.csv")

Submission saved → /kaggle/working/submissions/submission_v2.csv


'/kaggle/working/submissions/submission_v2.csv'

# Ensemble
## Train ResNet-50


In [9]:
from model import ResNetClassifier

model_resnet = ResNetClassifier(dropout=0.4)
trainer_resnet = Trainer(model_resnet, DEVICE, lr=1e-4, patience=5)
trainer_resnet.best_model_path = "/kaggle/working/models/best_model_resnet.pth"
trainer_resnet.fit(train_loader, val_loader, epochs=30)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s] 


Epoch 001 | Train Loss: 0.4955 | Val Loss: 0.0652
 Saved best model (val_loss=0.0652)


Epoch 002 | Train Loss: 0.1329 | Val Loss: 0.0510
 Saved best model (val_loss=0.0510)


Epoch 003 | Train Loss: 0.0503 | Val Loss: 0.0519


...

Epoch 020 | Train Loss: 0.0204 | Val Loss: 0.0478


Epoch 021 | Train Loss: 0.0085 | Val Loss: 0.0714


Epoch 022 | Train Loss: 0.0110 | Val Loss: 0.0335


Epoch 023 | Train Loss: 0.0087 | Val Loss: 0.0366


Epoch 024 | Train Loss: 0.0093 | Val Loss: 0.0386
Early stopping at epoch 24.


## Fine tune ResNet-50

In [10]:
model_resnet.unfreeze_backbone(layers_from_end=3)
trainer_resnet_ft = Trainer(model_resnet, DEVICE, lr=3e-5, patience=5)
trainer_resnet_ft.best_model_path = "/kaggle/working/models/best_model_resnet_v2.pth"
trainer_resnet_ft.fit(train_loader, val_loader, epochs=20)

Unfroze last 3 ResNet layers.


Epoch 001 | Train Loss: 0.0024 | Val Loss: 0.0308
 Saved best model (val_loss=0.0308)


Epoch 002 | Train Loss: 0.0016 | Val Loss: 0.0356


Epoch 003 | Train Loss: 0.0038 | Val Loss: 0.0304
 Saved best model (val_loss=0.0304)


Epoch 004 | Train Loss: 0.0009 | Val Loss: 0.0268
 Saved best model (val_loss=0.0268)


Epoch 005 | Train Loss: 0.0047 | Val Loss: 0.0308


Epoch 006 | Train Loss: 0.0014 | Val Loss: 0.0309


Epoch 007 | Train Loss: 0.0019 | Val Loss: 0.0288


Epoch 008 | Train Loss: 0.0012 | Val Loss: 0.0285


Epoch 009 | Train Loss: 0.0021 | Val Loss: 0.0303
Early stopping at epoch 9.


## Ensemble predict & Submit

In [11]:
from predictor import EnsemblePredictor

pred_effnet = Predictor(MaskClassifier(), DEVICE, "/kaggle/working/models/best_model_v2.pth")
pred_resnet = Predictor(ResNetClassifier(), DEVICE, "/kaggle/working/models/best_model_resnet_v2.pth")

ensemble = EnsemblePredictor(
    predictors=[pred_effnet, pred_resnet],
    weights=[0.3, 0.7]  # equal trust — adjust after seeing val losses
)

fnames, probs = ensemble.predict(test_loader, tta=True)
submitter = Submitter(output_dir="/kaggle/working/submissions")
submitter.save(fnames, probs, filename="submission_v3.csv")

Submission saved → /kaggle/working/submissions/submission_v3.csv


'/kaggle/working/submissions/submission_v3.csv'

In [12]:
from predictor import EnsemblePredictor

pred_effnet = Predictor(MaskClassifier(), DEVICE, "/kaggle/working/models/best_model_v2.pth")
pred_resnet = Predictor(ResNetClassifier(), DEVICE, "/kaggle/working/models/best_model_resnet_v2.pth")

ensemble = EnsemblePredictor(
    predictors=[pred_effnet, pred_resnet],
    weights=[0.1, 0.9]  # equal trust — adjust after seeing val losses
)

fnames, probs = ensemble.predict(test_loader, tta=True)
submitter = Submitter(output_dir="/kaggle/working/submissions")
submitter.save(fnames, probs, filename="submission_v4.csv")

Submission saved → /kaggle/working/submissions/submission_v4.csv


'/kaggle/working/submissions/submission_v4.csv'

In [13]:
from sklearn.metrics import log_loss
from predictor import EnsemblePredictor, Predictor
from model import MaskClassifier, ResNetClassifier

# Load both models
pred_effnet = Predictor(MaskClassifier(), DEVICE, "/kaggle/working/models/best_model_v2.pth")
pred_resnet = Predictor(ResNetClassifier(), DEVICE, "/kaggle/working/models/best_model_resnet_v2.pth")

# Get true val labels
true_labels = loader.val_df["target"].values

# Test different weight combinations
weight_combos = [(0.3, 0.7), (0.2, 0.8), (0.1, 0.9), (0.15, 0.85)]

print(f"{'EfficientNet':<15} {'ResNet':<10} {'Val Log Loss':<15}")
print("-" * 40)

best_loss = float("inf")
best_weights = None

for eff_w, res_w in weight_combos:
    ensemble = EnsemblePredictor(
        predictors=[pred_effnet, pred_resnet],
        weights=[eff_w, res_w]
    )
    fnames, probs = ensemble.predict(val_loader, tta=False)
    loss = log_loss(true_labels, probs)
    print(f"{eff_w:<15} {res_w:<10} {loss:.4f}")
    
    if loss < best_loss:
        best_loss = loss
        best_weights = (eff_w, res_w)

print(f"\nBest weights → EfficientNet: {best_weights[0]} | ResNet: {best_weights[1]} | Val Loss: {best_loss:.4f}")

EfficientNet    ResNet     Val Log Loss   
----------------------------------------
0.3             0.7        0.0291
0.2             0.8        0.0262
0.1             0.9        0.0246
0.15            0.85       0.0252

Best weights → EfficientNet: 0.1 | ResNet: 0.9 | Val Loss: 0.0246


TRAIN RESNET101

In [14]:
import sys
sys.path.append("/kaggle/input/datasets/moanlobago/3mry5-spotthemaskdata/3mry5_SpotTheMask/3mry5_SpotTheMask/src")

from model import ResNet101Classifier
from trainer import Trainer

DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model_resnet101 = ResNet101Classifier(dropout=0.4)
trainer_101 = Trainer(model_resnet101, DEVICE, lr=1e-4, patience=5)
trainer_101.best_model_path = "/kaggle/working/models/best_model_resnet101.pth"
trainer_101.fit(train_loader, val_loader, epochs=30)

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 186MB/s]  


Epoch 001 | Train Loss: 0.4332 | Val Loss: 0.0867
 Saved best model (val_loss=0.0867)


Epoch 002 | Train Loss: 0.0956 | Val Loss: 0.0579
 Saved best model (val_loss=0.0579)


Epoch 003 | Train Loss: 0.0449 | Val Loss: 0.0752


...

Epoch 008 | Train Loss: 0.0114 | Val Loss: 0.0327


Epoch 009 | Train Loss: 0.0108 | Val Loss: 0.0350


Epoch 010 | Train Loss: 0.0055 | Val Loss: 0.0367


Epoch 011 | Train Loss: 0.0061 | Val Loss: 0.0378


Epoch 012 | Train Loss: 0.0074 | Val Loss: 0.0352
Early stopping at epoch 12.


In [15]:
model_resnet101.unfreeze_backbone(layers_from_end=3)
trainer_101_ft = Trainer(model_resnet101, DEVICE, lr=3e-5, patience=5)
trainer_101_ft.best_model_path = "/kaggle/working/models/best_model_resnet101_v2.pth"
trainer_101_ft.fit(train_loader, val_loader, epochs=20)

Unfroze last 3 ResNet101 layers.


Epoch 001 | Train Loss: 0.0119 | Val Loss: 0.0325
 Saved best model (val_loss=0.0325)


Epoch 002 | Train Loss: 0.0077 | Val Loss: 0.0234
 Saved best model (val_loss=0.0234)


Epoch 003 | Train Loss: 0.0062 | Val Loss: 0.0291


Epoch 004 | Train Loss: 0.0033 | Val Loss: 0.0384


Epoch 005 | Train Loss: 0.0035 | Val Loss: 0.0302


Epoch 006 | Train Loss: 0.0055 | Val Loss: 0.0255


Epoch 007 | Train Loss: 0.0051 | Val Loss: 0.0261
Early stopping at epoch 7.


In [16]:
pred_101 = Predictor(ResNet101Classifier(), DEVICE, "/kaggle/working/models/best_model_resnet101_v2.pth")
fnames, probs = pred_101.predict(test_loader, tta=True)
submitter = Submitter(output_dir="/kaggle/working/submissions")
submitter.save(fnames, probs, filename="submission_resnet101.csv")

Submission saved → /kaggle/working/submissions/submission_resnet101.csv


'/kaggle/working/submissions/submission_resnet101.csv'